In [1]:
import os
import rasterio as rio
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.path as mpth
from pathlib import Path


import Functions
import importlib

importlib.reload(Functions)

output_folder = Path('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents')


In [8]:
app_path = Functions.get_input_path() / 'App'
input_path = app_path / 'Documents' / 'csv'

path: /home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA


Collecting the backscatter data from a csv

In [9]:
sigma_TM = pd.DataFrame(pd.read_csv(input_path / 'TimeSeries_sigma.csv'))
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
date_S1 = list(pd.unique(pd.to_datetime(sigma_TM['Date'])))
sigma_TM.drop(sigma_TM[sigma_TM['mean'] == 0].index,inplace=True)

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date'])

display(sigma_TM)

,Code,Date_Band,mean,std,Date,Band,Name,Type
1298,1545384,20170104_VH,-15.490180,1.038437,2017-01-04,VH,DB_SB2_C,Potatoes
1300,1545384,20170110_VH,-14.146261,1.136036,2017-01-10,VH,DB_SB2_C,Potatoes
1302,1545384,20170116_VH,-20.622658,0.922879,2017-01-16,VH,DB_SB2_C,Potatoes
1304,1545384,20170122_VH,-22.050701,0.694627,2017-01-22,VH,DB_SB2_C,Potatoes
1306,1545384,20170128_VH,-16.818918,0.731224,2017-01-28,VH,DB_SB2_C,Potatoes
...,...,...,...,...,...,...,...,...
935,2278835,20171206_VV,-6.205426,0.920910,2017-12-06,VV,PVD-M-C,Corn
937,2278835,20171212_VV,-11.775089,0.645342,2017-12-12,VV,PVD-M-C,Corn
939,2278835,20171218_VV,-5.921874,1.043583,2017-12-18,VV,PVD-M-C,Corn
941,2278835,20171224_VV,-6.724238,1.070252,2017-12-24,VV,PVD-M-C,Corn


Fixed the dates of the moisture: shifted the dates of the moisture to the closest (previuos) S1 backscatter date to have the data at the same date.

In [10]:
moisture = pd.read_excel('/home/frank/Desktop/f.chiapperino_local'
                        '/VALENCIA/SCUOLA/App/Data/Ground_Campaign/Flevoland_data/Data_25_fields/Average_Soil_moisture_N.xlsx',
                        header=0)

moisture.set_index('Code',inplace=True)
moisture = moisture[moisture.index.notna()]
moisture_date = list(pd.to_datetime(moisture.columns))

df_date_moist = pd.DataFrame({'original_date': moisture_date}).sort_values('original_date')
df_date_S1 = pd.DataFrame({'ref_date': date_S1}).sort_values('ref_date')

new_date_moist = pd.merge_asof(df_date_moist, df_date_S1, 
                                left_on='original_date',
                                right_on='ref_date', direction='backward')
moisture.columns = pd.to_datetime(new_date_moist['ref_date'])

moisture = moisture.reset_index().melt(id_vars='Code', var_name='Date', value_name='Moist_situ')
display(moisture)


,Code,Date,Moist_situ
0,1841225.0,2017-05-16 00:00:00,18.7375
1,1553694.0,2017-05-16 00:00:00,19.1125
2,1896343.0,2017-05-16 00:00:00,NaN
3,1841224.0,2017-05-16 00:00:00,14.7875
4,1841223.0,2017-05-16 00:00:00,16.3375
...,...,...,...
315,1698168.0,2017-09-13 00:00:00,NaN
316,1697689.0,2017-09-13 00:00:00,NaN
317,1697690.0,2017-09-13 00:00:00,NaN
318,2041694.0,2017-09-13 00:00:00,NaN


In [11]:
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
moisture['Date'] = pd.to_datetime(moisture['Date'])

Try the aplha method: $$SSM_{i+1} \approx \frac{\sigma_{0}^{i+1}}{\sigma_{0}^{i}} SSM_{i}$$

In [12]:
sigma_TM = pd.merge(
    sigma_TM, 
    moisture, 
    on=['Code', 'Date'], 
    how='right' 
)

calculated the difference between the backscatter at (t+1) and at (t). Then converted in linear scale. At the end grouped everything by the code of the field.

In [13]:
sigma_TM.dropna(subset='Moist_situ' ,inplace=True)
sigma_TM['diff_mean'] = -sigma_TM.groupby(['Code', 'Band'])['mean'].diff(periods=-1)
sigma_TM['diff_lin'] = 10 ** (sigma_TM['diff_mean']/10)
sigma_TM.insert(len(sigma_TM.columns)-1, 'Moist_situ', sigma_TM.pop('Moist_situ'))

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date']).reset_index(drop=True)
display(sigma_TM[sigma_TM['Code'] == 1698168])

,Code,Date_Band,mean,std,Date,Band,Name,Type,diff_mean,diff_lin,Moist_situ
120,1698168,20170516_VH,-24.783758,2.422116,2017-05-16,VH,TK_MA_C,Corn,-0.104262,0.976279,23.7625
121,1698168,20170522_VH,-24.888020,2.299481,2017-05-22,VH,TK_MA_C,Corn,1.309016,1.351766,17.4250
122,1698168,20170528_VH,-23.579004,2.071898,2017-05-28,VH,TK_MA_C,Corn,1.722257,1.486708,21.3375
123,1698168,20170603_VH,-21.856747,0.978037,2017-06-03,VH,TK_MA_C,Corn,2.914460,1.956348,28.9375
124,1698168,20170627_VH,-18.942287,1.170357,2017-06-27,VH,TK_MA_C,Corn,0.875977,1.223482,29.6375
125,1698168,20170709_VH,-18.066310,1.087744,2017-07-09,VH,TK_MA_C,Corn,1.120675,1.294397,34.6500
126,1698168,20170715_VH,-16.945635,1.326007,2017-07-15,VH,TK_MA_C,Corn,NaN,NaN,34.4000
127,1698168,20170516_VV,-15.506469,1.663272,2017-05-16,VV,TK_MA_C,Corn,-0.168909,0.961854,23.7625
128,1698168,20170522_VV,-15.675378,1.625232,2017-05-22,VV,TK_MA_C,Corn,0.226313,1.053492,17.4250
129,1698168,20170528_VV,-15.449065,1.285863,2017-05-28,VV,TK_MA_C,Corn,1.091595,1.285759,21.3375


The alpha method: multiplied the ratio of backscatter (σ(t+1) / σ(t)). This ration is placed onthe same row of the SSM (t+1): for this reason, to calculate the SSM (t) we use ratio and the SSm at t-1

In [14]:
def alpha_recursion(ssm_arr, ratio_arr, win=4):

    ssm_ret = ssm_arr.copy().astype(float)
    
    for b in range(0, len(ssm_ret), win):
        for passo in range(1, win):
    
            i = b + passo
            if i >= len(ssm_ret):
                break
            if not np.isnan(ssm_ret[i-1]) and not np.isnan(ratio_arr[i]):
                ssm_ret[i] = ratio_arr[i-1] * ssm_ret[i-1]
            else:
                continue
    
    return ssm_ret

sigma_TM['SSM_retrieved'] = sigma_TM.groupby(['Code', 'Band'], group_keys=False).apply(
    lambda g: pd.Series(alpha_recursion(g['Moist_situ'].values, g['diff_lin'].values), index=g.index)
)

In [15]:
display(sigma_TM[sigma_TM['Code']==1698168])

,Code,Date_Band,mean,std,Date,Band,Name,Type,diff_mean,diff_lin,Moist_situ,SSM_retrieved
120,1698168,20170516_VH,-24.783758,2.422116,2017-05-16,VH,TK_MA_C,Corn,-0.104262,0.976279,23.7625,23.762500
121,1698168,20170522_VH,-24.888020,2.299481,2017-05-22,VH,TK_MA_C,Corn,1.309016,1.351766,17.4250,23.198822
122,1698168,20170528_VH,-23.579004,2.071898,2017-05-28,VH,TK_MA_C,Corn,1.722257,1.486708,21.3375,31.359384
123,1698168,20170603_VH,-21.856747,0.978037,2017-06-03,VH,TK_MA_C,Corn,2.914460,1.956348,28.9375,46.622250
124,1698168,20170627_VH,-18.942287,1.170357,2017-06-27,VH,TK_MA_C,Corn,0.875977,1.223482,29.6375,29.637500
125,1698168,20170709_VH,-18.066310,1.087744,2017-07-09,VH,TK_MA_C,Corn,1.120675,1.294397,34.6500,36.260957
126,1698168,20170715_VH,-16.945635,1.326007,2017-07-15,VH,TK_MA_C,Corn,NaN,NaN,34.4000,34.400000
127,1698168,20170516_VV,-15.506469,1.663272,2017-05-16,VV,TK_MA_C,Corn,-0.168909,0.961854,23.7625,23.762500
128,1698168,20170522_VV,-15.675378,1.625232,2017-05-22,VV,TK_MA_C,Corn,0.226313,1.053492,17.4250,22.856053
129,1698168,20170528_VV,-15.449065,1.285863,2017-05-28,VV,TK_MA_C,Corn,1.091595,1.285759,21.3375,24.078672


As it mentions the article (Sentinel-1 Sensitivity to Soil Moisture at High Incidence Angle and the Impact on Retrieval Over Seasonal Crops, Davide Palmisano), the a priori datas were excluded from the linear regression: here they are excluded from the dataset.

In [16]:
sigma_TM = sigma_TM.loc[sigma_TM['Moist_situ'] != sigma_TM['SSM_retrieved']]
display(sigma_TM)

,Code,Date_Band,mean,std,Date,Band,Name,Type,diff_mean,diff_lin,Moist_situ,SSM_retrieved
1,1553694,20170522_VH,-22.459312,1.242794,2017-05-22,VH,DB_SB2_C,Beets,4.769868,2.999071,16.2500,27.680842
2,1553694,20170603_VH,-17.689444,1.524553,2017-06-03,VH,DB_SB2_C,Beets,3.346001,2.160728,27.5125,83.016821
3,1553694,20170609_VH,-14.343443,1.547611,2017-06-09,VH,DB_SB2_C,Beets,-1.950233,0.638229,15.9500,179.376770
5,1553694,20170627_VH,-16.053875,1.619533,2017-06-27,VH,DB_SB2_C,Beets,1.319586,1.355060,12.0125,7.146401
6,1553694,20170703_VH,-14.734289,1.456309,2017-07-03,VH,DB_SB2_C,Beets,-1.610401,0.690176,27.4625,9.683804
...,...,...,...,...,...,...,...,...,...,...,...,...
468,2278835,20170703_VV,-8.163950,1.054214,2017-07-03,VV,PVD-M-C,Corn,-3.074975,0.492609,25.1500,42.173819
469,2278835,20170709_VV,-11.238925,0.937927,2017-07-09,VV,PVD-M-C,Corn,-0.110318,0.974918,26.2750,20.775210
471,2278835,20170727_VV,-11.496948,0.947724,2017-07-27,VV,PVD-M-C,Corn,0.414126,1.100050,29.8125,27.764480
472,2278835,20170814_VV,-11.082822,0.921020,2017-08-14,VV,PVD-M-C,Corn,0.046305,1.010719,27.2500,30.542328


Regression between the in-situ results and the SSM-retrieved with the alpha method for the single fields.

In [17]:
from scipy import stats

results_regression = []

for (codice, banda, name,typ), gruppo in sigma_TM.groupby(['Code', 'Band', 'Name', 'Type']):   

    moist_situ = gruppo['Moist_situ'].values
    SSM_ret = gruppo['SSM_retrieved'].values

    if len(moist_situ) > 2:

        slope, intercept, r_value, p_value, std_err = stats.linregress(moist_situ, SSM_ret)        
        y_pred = slope * moist_situ + intercept
        
        residui = SSM_ret - y_pred
        
        rmsre = np.sqrt(np.mean(residui ** 2))
        
        results_regression.append({
            'Code': codice,
            'Name': name,
            'Band': banda,
            'Type': typ,
            'N_points': len(moist_situ),
            'R_Pearson': r_value,
            'R_2': r_value ** 2,
            'P_value': p_value,
            'Slope': slope,
            'Intercept': intercept,
            'RMSRE': rmsre
        })

df_statistiche = pd.DataFrame(results_regression)
df_statistiche.sort_values('R_2', ascending=False, inplace=True)
display(df_statistiche.head())

filename = "moisture_reg_xfield.csv"
output_path = output_folder / 'csv' / filename

df_statistiche.to_csv(output_path ,index=False, sep=';', decimal='.')

,Code,Name,Band,Type,N_points,R_Pearson,R_2,P_value,Slope,Intercept,RMSRE
13,1698168,TK_MA_C,VV,Corn,4,0.982671,0.965643,0.017329,0.916125,5.640185,1.152881
8,1697689,SB7_ER_C,VH,Grassland,3,-0.903068,0.815532,0.282619,-3.214208,166.644159,4.603430
35,2081268,RBW_A_C,VV,Potatoes,10,0.783496,0.613867,0.007333,1.009428,-1.029010,4.093891
9,1697689,SB7_ER_C,VV,Grassland,3,-0.769286,0.591801,0.441224,-3.513688,184.274942,8.787771
12,1698168,TK_MA_C,VH,Corn,4,0.716689,0.513644,0.283311,0.910814,11.054888,5.912968


Here a new regression was calculaded, excluding the outliers (values that are 3*stdv away from the value found with the regression).

In [18]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
from matplotlib.backends.backend_pdf import PdfPages

results_regression_f = []

output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'Regression_Moisture_xField.pdf'


with PdfPages(pdf_filename) as pdf:

    for i, (_, row) in enumerate(df_statistiche.iterrows()):

        codice = row['Code']
        banda = row['Band']
        tipo = row['Type']
        
        dati_gruppo = sigma_TM[(sigma_TM['Code'] == codice) & (sigma_TM['Band'] == banda)].sort_values('Date')
        
        x_in_situ = dati_gruppo['Moist_situ'].values
        y_calcolata = dati_gruppo['SSM_retrieved'].values
        date_asse = dati_gruppo['Date'].values
        
        if len(x_in_situ) > 2:
            
            # OUTLIER SEARCH
            y_pred_grezza = row['Slope'] * x_in_situ + row['Intercept']
            residui = y_calcolata - y_pred_grezza
            
            # THRESHOLD
            soglia = 3 * row['RMSRE']
            
            mask_outlier = np.abs(residui) > soglia
            mask_inlier = np.abs(residui) <= soglia
            n_outliers = np.sum(mask_outlier)
            
            x_puliti = x_in_situ[mask_inlier]
            y_puliti = y_calcolata[mask_inlier]
            
            if len(x_puliti) > 2:
                
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
                
                # REGRSSION W/ OUTLIER
                slope_f, intercept_f, r_f, p_f, std_err = stats.linregress(x_puliti, y_puliti)
                r2_pulito = r_f ** 2

                residui = y_puliti - x_puliti
                
                rmsre_f = np.sqrt(np.mean(residui ** 2))

                results_regression_f.append({
                    'Code': codice,
                    'Band': banda,
                    'Type': tipo,
                    'N_points': len(x_puliti),
                    'R_Pearson': r_f,
                    'R_2': r2_pulito,
                    'P_value': p_f,
                    'Slope': slope_f,
                    'Intercept': intercept_f,
                    'RMSRE': rmsre_f
                })

                # =================================================================
                # SUBPLOT 1: SCATTER PLOT & REGRESSIONE (ax1)
                # =================================================================
                
                # 1. Disegniamo i punti validi (Blue/Teal)
                sns.scatterplot(x=x_puliti, y=y_puliti, 
                                color='#1f77b4', s=60, label='Valid Data', alpha=0.8, ax=ax1)
                
                # 2. Disegniamo gli outlier rimossi (Rosso con 'X')
                if n_outliers > 0:
                    sns.scatterplot(x=x_in_situ[mask_outlier], y=y_calcolata[mask_outlier], 
                                    marker='x', color="#ca4828", s=60, label='Outliers', alpha=0.8, ax=ax1)
                
                # 3. Disegniamo la retta di regressione PULITA
                x_linea = np.linspace(x_puliti.min(), x_puliti.max(), 100)
                y_linea = slope_f * x_linea + intercept_f

                sns.lineplot(x=x_linea, y=y_linea, color="#ca4828", 
                             label=f'Cleaned Fit\n$R^2$ without outlier = {r2_pulito:.3f}', alpha=0.8, ax=ax1)
                
                ax1.fill_between(x_linea, 
                                 y_linea - std_err, 
                                 y_linea + std_err, 
                                 color="#ca4828", alpha=0.15, label='1 Std. Dev.')

                # Linea di riferimento ideale 1:1
                limiti = [min(x_in_situ.min(), y_calcolata.min()), max(x_in_situ.max(), y_calcolata.max())]
                ax1.plot(limiti, limiti, color='gray', linestyle=':', alpha=0.5, label='Ideal 1:1')
                
                # Formattazione pannello 1
                ax1.set_title(f"{codice} - {banda} - {tipo}\n($R^2$ raw: {row['R_2']:.3f})",
                              fontsize=11, fontweight='bold')
                ax1.set_xlabel("SSM In Situ ($m^3/m^3$)", fontsize=10)
                ax1.set_ylabel("SSM Retr ($m^3/m^3$)", fontsize=10)
                ax1.legend(loc='upper left', fontsize=9)
                ax1.grid(True, linestyle='--', alpha=0.5)

                # =================================================================
                # SUBPLOT 2: ANDAMENTO TEMPORALE (ax2)
                # =================================================================
                
                # Grafico linea + marker per l'umidità In Situ (usiamo il verde per distinguerlo bene)
                sns.lineplot(x=date_asse, y=x_in_situ, color='#2ca02c', marker='o', 
                             linewidth=1.8, label='SSM In Situ', ax=ax2)
                
                # Grafico linea + marker per l'umidità Stimata (riprendiamo il blu dei dati validi dello scatter)
                sns.lineplot(x=date_asse, y=y_calcolata, color='#1f77b4', marker='^', linestyle='--',
                             linewidth=1.5, label='SSM Retrieved', ax=ax2)
                
                # Opzionale ma consigliato: marchiamo con una 'X' rossa gli outlier anche sulla linea temporale della stima
                if n_outliers > 0:
                    ax2.scatter(date_asse[mask_outlier], y_calcolata[mask_outlier], 
                                marker='x', color="#ca4828", s=70, zorder=5, label='Outliers Identified')

                # Formattazione pannello 2
                ax2.set_title("Time Series Humidity", fontsize=11, fontweight='bold')
                ax2.set_xlabel("Date", fontsize=10)
                ax2.set_ylabel("SSM ($m^3/m^3$)", fontsize=10)
                ax2.legend(loc='upper right', fontsize=9)
                ax2.grid(True, linestyle='--', alpha=0.5)
                
                # Ruotiamo le date sull'asse X per non farle sovrapporre
                ax2.tick_params(axis='x', rotation=30)
                
                # Compattiamo il layout globale prima del salvataggio
                plt.tight_layout()
                
                # Salva la figura a due pannelli nella pagina corrente del PDF
                pdf.savefig()
                plt.close()




In [19]:
results_regression_f = pd.DataFrame(results_regression_f)
results_regression_f.sort_values('R_2', ascending=False, inplace=True)
display(results_regression_f)

filename = "moisture_reg_xfield_no_outlier.csv"
output_path = output_folder / 'csv' / filename

results_regression_f.to_csv(output_path ,index=False, sep=';', decimal='.')

,Code,Band,Type,N_points,R_Pearson,R_2,P_value,Slope,Intercept,RMSRE
0,1698168,VV,Corn,4,0.982671,0.965643,0.017329,0.916125,5.640185,3.721624
1,1697689,VH,Grassland,3,-0.903068,0.815532,0.282619,-3.214208,166.644159,23.628820
2,2081268,VV,Potatoes,10,0.783496,0.613867,0.007333,1.009428,-1.029010,4.172047
3,1697689,VV,Grassland,3,-0.769286,0.591801,0.441224,-3.513688,184.274942,31.102555
4,1698168,VH,Corn,4,0.716689,0.513644,0.283311,0.910814,11.054888,10.596235
5,2081887,VH,Corn,7,-0.683683,0.467423,0.090344,-3.512100,138.840381,39.653170
6,1936133,VH,Potatoes,9,-0.682008,0.465135,0.043005,-3.509535,101.382494,28.335156
7,1601502,VV,Beets,10,0.676117,0.457134,0.031840,1.022514,-0.717176,6.831641
8,2081268,VH,Potatoes,10,0.662578,0.439009,0.036822,0.980328,3.501312,6.426050
9,2081887,VV,Corn,7,-0.662150,0.438442,0.105140,-2.530215,108.444592,30.723568


Here tha same thing is done but tthe regression was calculated on all the values of SSM available for the same type of field (eg wheat, grass, potatoes...).

In [20]:
results_regression_group = []

for ( banda, typ ), gruppo in sigma_TM.groupby(['Band', 'Type']):   

    moist_situ = gruppo['Moist_situ'].values
    SSM_ret = gruppo['SSM_retrieved'].values

    if len(moist_situ) > 2:

        slope, intercept, r_value, p_value, std_err = stats.linregress(moist_situ, SSM_ret)        
        y_pred = slope * moist_situ + intercept
        
        residui = SSM_ret - y_pred
        
        rmsre = np.sqrt(np.mean(residui ** 2))
        
        results_regression_group.append({
            'Band': banda,
            'Type': typ,
            'N_points': len(moist_situ),
            'R_Pearson': r_value,
            'R_2': r_value ** 2,
            'P_value': p_value,
            'Slope': slope,
            'Intercept': intercept,
            'RMSRE': rmsre
        })

results_regression_group = pd.DataFrame(results_regression_group)
results_regression_group.sort_values('R_2', ascending=False, inplace=True)
display(results_regression_group)

filename = "moisture_reg_group.csv"
output_path = output_folder / 'csv' / filename

results_regression_group.to_csv(output_path ,index=False, sep=';', decimal='.')

,Band,Type,N_points,R_Pearson,R_2,P_value,Slope,Intercept,RMSRE
7,VV,Grassland,19,0.577914,0.333984,0.009554,1.307051,1.304684,13.357593
2,VH,Grassland,19,0.551007,0.303609,0.014480,1.187904,5.440790,13.019870
4,VH,Wheat,34,0.333571,0.111269,0.053867,0.370956,16.999088,8.203569
3,VH,Potatoes,34,-0.216126,0.046710,0.219577,-1.145543,54.911621,28.060914
1,VH,Corn,29,-0.159540,0.025453,0.408423,-0.944651,63.828171,33.361539
8,VV,Potatoes,34,-0.138134,0.019081,0.435939,-0.400132,30.979165,15.556240
0,VH,Beets,37,-0.085686,0.007342,0.614088,-0.550788,51.262832,39.100896
5,VV,Beets,37,-0.068688,0.004718,0.686251,-0.309776,37.354438,27.469751
6,VV,Corn,29,-0.028906,0.000836,0.881675,-0.075760,35.220344,14.952579
9,VV,Wheat,34,0.017370,0.000302,0.922329,0.022609,31.484516,10.183703


In [21]:
results_regression_group_f = []

pdf_filename = output_folder / 'Regression_Moisture_Group.pdf'

with PdfPages(pdf_filename) as pdf:

    for i, (_, row) in enumerate(results_regression_group.iterrows()):

        banda = row['Band']
        tipo = row['Type']
        
        dati_gruppo = sigma_TM[(sigma_TM['Type'] == tipo) & (sigma_TM['Band'] == banda)].sort_values('Date')
        
        x_in_situ = dati_gruppo['Moist_situ'].values
        y_calcolata = dati_gruppo['SSM_retrieved'].values
        date_asse = dati_gruppo['Date'].values
        
        if len(x_in_situ) > 2:
            
            # OUTLIER SEARCH
            y_pred_grezza = row['Slope'] * x_in_situ + row['Intercept']
            residui = y_calcolata - y_pred_grezza
            
            # THRESHOLD
            soglia = 3 * row['RMSRE']
            
            mask_outlier = np.abs(residui) > soglia
            mask_inlier = np.abs(residui) <= soglia
            n_outliers = np.sum(mask_outlier)
            
            x_puliti = x_in_situ[mask_inlier]
            y_puliti = y_calcolata[mask_inlier]
            
            if len(x_puliti) > 2:
                
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
                
                # REGRSSION W/ OUTLIER
                slope_f, intercept_f, r_f, p_f, std_err = stats.linregress(x_puliti, y_puliti)
                r2_pulito = r_f ** 2

                residui = y_puliti - x_puliti
                
                rmsre = np.sqrt(np.mean(residui ** 2))

                results_regression_group_f.append({
                    'Band': banda,
                    'Type': tipo,
                    'N_points': len(x_puliti),
                    'R_Pearson': r_f,
                    'R_2': r2_pulito,
                    'P_value': p_f,
                    'Slope': slope_f,
                    'Intercept': intercept_f,
                    'RMSRE': rmsre
                })

                # =================================================================
                # SUBPLOT 1: SCATTER PLOT & REGRESSIONE (ax1)
                # =================================================================
                
                # 1. Disegniamo i punti validi (Blue/Teal)
                sns.scatterplot(x=x_puliti, y=y_puliti, 
                                color='#1f77b4', s=60, label='Valid Data', alpha=0.8, ax=ax1)
                
                # 2. Disegniamo gli outlier rimossi (Rosso con 'X')
                if n_outliers > 0:
                    sns.scatterplot(x=x_in_situ[mask_outlier], y=y_calcolata[mask_outlier], 
                                    marker='x', color="#ca4828", s=60, label='Outliers', alpha=0.8, ax=ax1)
                
                # 3. Disegniamo la retta di regressione PULITA
                x_linea = np.linspace(x_puliti.min(), x_puliti.max(), 100)
                y_linea = slope_f * x_linea + intercept_f

                sns.lineplot(x=x_linea, y=y_linea, color="#ca4828", 
                             label=f'Cleaned Fit\n$R^2$ without outlier = {r2_pulito:.3f}', alpha=0.8, ax=ax1)
                
                ax1.fill_between(x_linea, 
                                 y_linea - std_err, 
                                 y_linea + std_err, 
                                 color="#ca4828", alpha=0.15, label='1 Std. Dev.')

                # Linea di riferimento ideale 1:1
                limiti = [min(x_in_situ.min(), y_calcolata.min()), max(x_in_situ.max(), y_calcolata.max())]
                ax1.plot(limiti, limiti, color='gray', linestyle=':', alpha=0.5, label='Ideal 1:1')
                
                # Formattazione pannello 1
                ax1.set_title(f"{tipo} - {banda}\n($R^2$ raw: {row['R_2']:.3f})",
                              fontsize=11, fontweight='bold')
                ax1.set_xlabel("SSM In Situ ($m^3/m^3$)", fontsize=10)
                ax1.set_ylabel("SSM Retr ($m^3/m^3$)", fontsize=10)
                ax1.legend(loc='upper left', fontsize=9)
                ax1.grid(True, linestyle='--', alpha=0.5)

                # =================================================================
                # SUBPLOT 2: ANDAMENTO TEMPORALE (ax2)
                # =================================================================
                
                # Grafico linea + marker per l'umidità In Situ (usiamo il verde per distinguerlo bene)
                sns.lineplot(x=date_asse, y=x_in_situ, color='#2ca02c', marker='o', 
                             linewidth=1.8, label='SSM In Situ', ax=ax2)
                
                # Grafico linea + marker per l'umidità Stimata (riprendiamo il blu dei dati validi dello scatter)
                sns.lineplot(x=date_asse, y=y_calcolata, color='#1f77b4', marker='^', linestyle='--',
                             linewidth=1.5, label='SSM Retrieved', ax=ax2)
                
                # Opzionale ma consigliato: marchiamo con una 'X' rossa gli outlier anche sulla linea temporale della stima
                if n_outliers > 0:
                    ax2.scatter(date_asse[mask_outlier], y_calcolata[mask_outlier], 
                                marker='x', color="#ca4828", s=70, zorder=5, label='Outliers Identified')

                # Formattazione pannello 2
                ax2.set_title("Time Series Humidity", fontsize=11, fontweight='bold')
                ax2.set_xlabel("Date", fontsize=10)
                ax2.set_ylabel("SSM ($m^3/m^3$)", fontsize=10)
                ax2.legend(loc='upper right', fontsize=9)
                ax2.grid(True, linestyle='--', alpha=0.5)
                
                # Ruotiamo le date sull'asse X per non farle sovrapporre
                ax2.tick_params(axis='x', rotation=30)
                
                # Compattiamo il layout globale prima del salvataggio
                plt.tight_layout()
                
                # Salva la figura a due pannelli nella pagina corrente del PDF
                pdf.savefig()
                plt.close()


In [22]:
results_regression_group_f = pd.DataFrame(results_regression_group_f)
results_regression_group_f.sort_values('R_2', ascending=False, inplace=True)
display(results_regression_group_f)

filename = "moisture_reg_group_no_outlier.csv"
output_path = output_folder / 'csv' / filename

results_regression_group_f.to_csv(output_path ,index=False, sep=';', decimal='.')

,Band,Type,N_points,R_Pearson,R_2,P_value,Slope,Intercept,RMSRE
0,VV,Grassland,19,0.577914,0.333984,0.009554,1.307051,1.304684,16.591637
1,VH,Grassland,19,0.551007,0.303609,0.014480,1.187904,5.440790,16.787386
7,VV,Beets,35,0.348014,0.121114,0.040497,0.822977,5.739129,13.522722
2,VH,Wheat,34,0.333571,0.111269,0.053867,0.370956,16.999088,9.955043
8,VV,Corn,28,0.117222,0.013741,0.552474,0.208489,26.217076,13.116087
3,VH,Potatoes,33,-0.058649,0.003440,0.745787,-0.182272,29.946452,18.265227
5,VV,Potatoes,33,0.030892,0.000954,0.864491,0.064431,18.939062,12.356142
6,VH,Beets,36,0.028796,0.000829,0.867595,0.151015,31.467997,34.520941
9,VV,Wheat,34,0.017370,0.000302,0.922329,0.022609,31.484516,15.798528
4,VH,Corn,28,-0.016668,0.000278,0.932910,-0.060574,37.358703,24.461722
